# Deutsche Bahn Historical Delay Data Profiling — January to July 2026

## Purpose

This notebook profiles the monthly Deutsche Bahn historical delay Parquet files for
**January through July 2026**. 

The notebook is intentionally **development-time profiling**, not production
transformation logic.

> **Working rule:** observe → quantify → inspect examples → document a decision →
> implement the final semantic rule later in dbt.

### Profiling window

- `data-2026-01.parquet`
- `data-2026-02.parquet`
- `data-2026-03.parquet`
- `data-2026-04.parquet`
- `data-2026-05.parquet`
- `data-2026-06.parquet`
- `data-2026-07.parquet`

### Main outcomes

1. Quantify volume, completeness, cardinality, and value distributions.
2. Establish the row grain and investigate repeated ride-stop observations.
3. Validate timestamp relationships and the meaning of the generic `time` field.
4. Test whether `delay_in_min` can be reproduced from planned/effective event times.
5. Characterize cancellations and context-dependent null patterns.
6. Turn findings into explicit candidate dbt transformations and dbt tests.
7. Confirm which dashboard questions can be supported by the modeled data.


## 1. Profiling questions

The profiling is organized around decisions that the shared dbt core will later need
to make.

| Area | Profiling question | Why it matters later |
|---|---|---|
| Grain | What exactly makes one observation unique? | Defines `int_observations_deduped` |
| Duplicate ride-stops | Are repeated ride-stop rows updates, snapshots, or exact repeats? | Defines dedup precedence |
| Completeness | Which nulls are structural and which look like quality problems? | Prevents invalid blanket `not_null` rules |
| Time | Does `time` correspond to arrival, departure, both, or neither? | Defines service date/hour semantics |
| Delay | Does published `delay_in_min` match arrival/departure timestamp differences? | Defines one shared delay rule |
| Cancellation | How do canceled observations behave in delay and timestamp fields? | Defines mart denominators and filters |
| Dimensions | Are EVA/station names, line numbers, and train types stable enough for marts? | Defines conformed dimensions |
| Sanity bounds | What do the delay tails look like? | Informs dbt anomaly tests without guessing thresholds |

No business rule is considered final until the relevant profiling evidence has been
reviewed.


In [1]:
from datetime import datetime, timezone
from html import escape
from pathlib import Path

from IPython.display import HTML, display

from pyspark import StorageLevel
from pyspark.sql import DataFrame, SparkSession, Window, functions as F
from pyspark.sql.types import StructType


## 2. Paths, profiling scope, and known source contract

The data catalog identified a 17-column physical schema. The list below is used only
as a **contract check**. The actual schema read from each file remains the source of
truth for profiling.


In [2]:
DATA_DIRECTORY = Path(
    "/opt/spark/work-dir/data/historical_delays"
)

PROFILE_MONTHS = [
    f"2026-{month:02d}"
    for month in range(1, 8)
]

DATA_PATHS = {
    period: DATA_DIRECTORY / f"data-{period}.parquet"
    for period in PROFILE_MONTHS
}

PROFILE_OUTPUT_PATH = Path(
    "/opt/spark/work-dir/artifacts/profiling/"
    "deutsche_bahn_2026_01_07"
)

EXPECTED_COLUMNS = [
    "station_name",
    "xml_station_name",
    "eva",
    "train_number",
    "line_number",
    "final_destination_station",
    "delay_in_min",
    "time",
    "is_canceled",
    "train_type",
    "train_line_ride_id",
    "train_line_station_num",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
    "id",
]

RAW_TIMESTAMP_COLUMNS = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]

NS_PER_MINUTE = 60_000_000_000

PROFILE_GENERATED_AT_UTC = datetime.now(
    timezone.utc
).isoformat()

missing_files = [
    str(path)
    for path in DATA_PATHS.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The profiling scope requires all January-July 2026 files. "
        "Missing:\n- " + "\n- ".join(missing_files)
    )

for period, path in DATA_PATHS.items():
    print(
        f"{period}: {path.name} "
        f"({path.stat().st_size / 1024**2:.2f} MB)"
    )


2026-01: data-2026-01.parquet (620.54 MB)
2026-02: data-2026-02.parquet (547.28 MB)
2026-03: data-2026-03.parquet (597.84 MB)
2026-04: data-2026-04.parquet (564.63 MB)
2026-05: data-2026-05.parquet (574.70 MB)
2026-06: data-2026-06.parquet (589.73 MB)
2026-07: data-2026-07.parquet (563.05 MB)


## 3. Reuse or create the Spark session

The monthly Parquet files store timestamp fields with nanosecond precision. As in the
data catalog, Spark is configured to expose unsupported Parquet nanosecond timestamps
as `long` values. Profiling-only decoded timestamp columns are then added separately.

The session timezone remains UTC for deterministic technical decoding. The business
interpretation of source timestamps as `Europe/Berlin` remains an item to verify with
upstream source logic; this notebook does not silently resolve that open question.    
      
> Defaults (1g executor heap) OOM-kill on this dataset size; sized to fit the 7.75 GiB Docker host.
> * .config("spark.driver.memory", "2g")
> * .config("spark.executor.memory", "4g")
> * .config("spark.executor.memoryOverhead", "1g")
> * .config("spark.sql.shuffle.partitions", "32")

In [3]:
SOURCE_TIME_ZONE = "Europe/Berlin"

try:
    spark
except NameError:
    spark = (
        SparkSession.builder
        .appName("deutsche-bahn-data-profiling-2026-01-07")
        .master("spark://spark-master:7077")
        .config("spark.driver.host", "spark-jupyter")
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.memory", "2g")
        .config("spark.executor.memory", "4g")
        .config("spark.executor.memoryOverhead", "1g")
        .config("spark.sql.shuffle.partitions", "32")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC",
)

spark.conf.set(
    "spark.sql.legacy.parquet.nanosAsLong",
    "true",
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print(
    "Technical decoding timezone:",
    spark.conf.get("spark.sql.session.timeZone"),
)
print("Intended business timezone:", SOURCE_TIME_ZONE)



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/09 19:58:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8
Spark master: spark://spark-master:7077
Technical decoding timezone: UTC
Intended business timezone: Europe/Berlin


### Small-table display helper

Profiling outputs are aggregated Spark DataFrames. The helper below collects only a
limited number of result rows and renders them as plain HTML, avoiding a runtime pandas
installation.


In [4]:
def display_spark_table(
    dataframe: DataFrame,
    title: str | None = None,
    limit: int = 100,
) -> None:
    rows = dataframe.limit(limit).collect()
    columns = dataframe.columns

    title_html = (
        f"<h4>{escape(title)}</h4>"
        if title
        else ""
    )

    header = "".join(
        f"<th>{escape(column)}</th>"
        for column in columns
    )

    body_rows = []
    for row in rows:
        cells = "".join(
            "<td>"
            + escape("" if value is None else str(value))
            + "</td>"
            for value in row
        )
        body_rows.append(f"<tr>{cells}</tr>")

    html = f"""
    {title_html}
    <div style="overflow-x:auto; max-height:560px; overflow-y:auto;">
      <table style="border-collapse:collapse; width:100%; font-size:13px;">
        <thead>
          <tr style="background:#f3f4f6; text-align:left;">
            {header}
          </tr>
        </thead>
        <tbody>
          {''.join(body_rows)}
        </tbody>
      </table>
    </div>
    <style>
      table th, table td {{
        border: 1px solid #d1d5db;
        padding: 6px 8px;
        vertical-align: top;
        white-space: normal;
      }}
    </style>
    """

    display(HTML(html))


## 4. Validate schema consistency across all seven months

Schema consistency was previously checked only for July 2024 and July 2026. Here
it is extended to every file in the profiling window before any multi-month analysis.

This check reads Parquet metadata rather than scanning every row.


In [5]:
monthly_schemas: dict[str, StructType] = {}

for period, path in DATA_PATHS.items():
    monthly_schemas[period] = spark.read.parquet(
        str(path)
    ).schema

reference_period = PROFILE_MONTHS[0]
reference_schema = monthly_schemas[reference_period]

schema_month_rows = []

for period in PROFILE_MONTHS:
    schema = monthly_schemas[period]
    actual_columns = [field.name for field in schema.fields]

    schema_month_rows.append(
        (
            period,
            len(actual_columns),
            schema == reference_schema,
            actual_columns == EXPECTED_COLUMNS,
            ", ".join(
                sorted(set(EXPECTED_COLUMNS) - set(actual_columns))
            ),
            ", ".join(
                sorted(set(actual_columns) - set(EXPECTED_COLUMNS))
            ),
        )
    )

schema_month_summary_df = spark.createDataFrame(
    schema_month_rows,
    (
        "period string, "
        "column_count int, "
        "matches_2026_01_schema boolean, "
        "matches_catalog_column_order boolean, "
        "missing_expected_columns string, "
        "unexpected_columns string"
    ),
).orderBy("period")

display_spark_table(
    schema_month_summary_df,
    "Schema contract by month",
)


period,column_count,matches_2026_01_schema,matches_catalog_column_order,missing_expected_columns,unexpected_columns
2026-01,17,True,True,,
2026-02,17,True,True,,
2026-03,17,True,True,,
2026-04,17,True,True,,
2026-05,17,True,True,,
2026-06,17,True,True,,
2026-07,17,True,True,,


In [6]:
reference_fields = {
    field.name: (
        field.dataType.simpleString(),
        field.nullable,
    )
    for field in reference_schema.fields
}

schema_field_rows = []

for period, schema in monthly_schemas.items():
    period_fields = {
        field.name: (
            field.dataType.simpleString(),
            field.nullable,
        )
        for field in schema.fields
    }

    for column_name in sorted(
        set(reference_fields) | set(period_fields)
    ):
        reference_definition = reference_fields.get(column_name)
        observed_definition = period_fields.get(column_name)

        schema_field_rows.append(
            (
                period,
                column_name,
                (
                    observed_definition[0]
                    if observed_definition
                    else None
                ),
                (
                    observed_definition[1]
                    if observed_definition
                    else None
                ),
                observed_definition == reference_definition,
            )
        )

schema_field_comparison_df = spark.createDataFrame(
    schema_field_rows,
    (
        "period string, "
        "column_name string, "
        "spark_data_type string, "
        "nullable boolean, "
        "matches_reference boolean"
    ),
)

schema_differences_df = (
    schema_field_comparison_df
    .filter(~F.col("matches_reference"))
    .orderBy("period", "column_name")
)

print(
    "Schema field differences across Jan-Jul 2026:",
    schema_differences_df.count(),
)

if schema_differences_df.limit(1).count() > 0:
    display_spark_table(
        schema_differences_df,
        "Schema differences",
    )


Schema field differences across Jan-Jul 2026: 0


## 5. Load the Jan–Jul 2026 profiling dataset

The seven files are read as one logical Spark DataFrame. Two profiling metadata fields
are added:

- `source_file` — physical Parquet file that produced the row;
- `source_month` — month parsed from the file name.

These are profiling metadata, not part of the 17-column source contract.


In [7]:
profile_paths = [
    str(DATA_PATHS[period])
    for period in PROFILE_MONTHS
]

historical_2026_raw = (
    spark.read.parquet(*profile_paths)
    .withColumn(
        "source_file",
        F.input_file_name(),
    )
    .withColumn(
        "source_month",
        F.regexp_extract(
            F.col("source_file"),
            r"data-(\d{4}-\d{2})\.parquet",
            1,
        ),
    )
)

print("Physical source columns:", len(EXPECTED_COLUMNS))
print("Profiling columns:", historical_2026_raw.columns)


Physical source columns: 17
Profiling columns: ['station_name', 'xml_station_name', 'eva', 'train_number', 'line_number', 'final_destination_station', 'delay_in_min', 'time', 'is_canceled', 'train_type', 'train_line_ride_id', 'train_line_station_num', 'arrival_planned_time', 'arrival_change_time', 'departure_planned_time', 'departure_change_time', 'id', 'source_file', 'source_month']


### Add decoded timestamp columns without replacing raw values

For each raw nanosecond field, a corresponding `*_ts` column is created. The original
`long` remains untouched so exact equality and recomputation checks can still use the
physical source value.


In [8]:
def add_decoded_timestamp_columns(
    dataframe: DataFrame,
    timestamp_columns: list[str],
) -> DataFrame:
    missing = sorted(
        set(timestamp_columns) - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            "Timestamp columns missing from schema: "
            f"{missing}"
        )

    transformed = dataframe

    for column_name in timestamp_columns:
        transformed = transformed.withColumn(
            f"{column_name}_ts",
            F.when(
                F.col(column_name).isNotNull(),
                F.timestamp_micros(
                    F.expr(
                        f"`{column_name}` DIV 1000"
                    )
                ),
            ).otherwise(
                F.lit(None).cast("timestamp")
            ),
        )

    return transformed


historical_2026 = add_decoded_timestamp_columns(
    historical_2026_raw,
    RAW_TIMESTAMP_COLUMNS,
)


## 6. Dataset volume and temporal coverage

This section validates the sizing assumption in the project plan and confirms that each
monthly file covers the intended service dates. Approximate distinct counts are used
for broad high-cardinality profiling; exact checks are reserved for key/grain analysis.


In [9]:
monthly_overview_df = (
    historical_2026
    .groupBy("source_month")
    .agg(
        F.count("*").alias("row_count"),
        F.min("time_ts").alias("minimum_time"),
        F.max("time_ts").alias("maximum_time"),
        F.countDistinct(
            F.to_date("time_ts")
        ).alias("distinct_service_dates"),
        F.approx_count_distinct(
            "eva",
            rsd=0.02,
        ).alias("approx_distinct_eva"),
        F.approx_count_distinct(
            "train_line_ride_id",
            rsd=0.02,
        ).alias("approx_distinct_rides"),
        F.approx_count_distinct(
            "train_number",
            rsd=0.02,
        ).alias("approx_distinct_train_numbers"),
        F.sum(
            F.when(F.col("is_canceled") == True, 1)
            .otherwise(0)
        ).alias("canceled_rows"),
    )
    .withColumn(
        "cancellation_rate_pct",
        F.round(
            100.0 * F.col("canceled_rows") / F.col("row_count"),
            4,
        ),
    )
)

file_size_df = spark.createDataFrame(
    [
        (
            period,
            DATA_PATHS[period].name,
            round(
                DATA_PATHS[period].stat().st_size / 1024**2,
                2,
            ),
        )
        for period in PROFILE_MONTHS
    ],
    "source_month string, file_name string, file_size_mb double",
)

monthly_overview_df = (
    monthly_overview_df
    .join(file_size_df, on="source_month", how="left")
    .select(
        "source_month",
        "file_name",
        "file_size_mb",
        "row_count",
        "minimum_time",
        "maximum_time",
        "distinct_service_dates",
        "approx_distinct_eva",
        "approx_distinct_rides",
        "approx_distinct_train_numbers",
        "canceled_rows",
        "cancellation_rate_pct",
    )
    .orderBy("source_month")
)

display_spark_table(
    monthly_overview_df,
    "Monthly volume and coverage",
)


26/08/09 20:00:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/09 20:00:16 WARN DAGScheduler: Broadcasting large task binary with size 1113.9 KiB
                                                                                

source_month,file_name,file_size_mb,row_count,minimum_time,maximum_time,distinct_service_dates,approx_distinct_eva,approx_distinct_rides,approx_distinct_train_numbers,canceled_rows,cancellation_rate_pct
2026-01,data-2026-01.parquet,620.54,15582748,2026-01-01 00:00:00,2026-01-31 23:59:00,31,5400,79064,49353,904186,5.8025
2026-02,data-2026-02.parquet,547.28,13721520,2026-02-01 00:00:00,2026-02-28 23:59:00,28,5400,84595,50650,503781,3.6715
2026-03,data-2026-03.parquet,597.84,15016329,2026-03-01 00:00:00,2026-03-31 23:59:00,31,5381,88715,51685,464261,3.0917
2026-04,data-2026-04.parquet,564.63,14149788,2026-04-01 00:00:00,2026-04-30 23:59:00,30,5359,82871,50373,447029,3.1593
2026-05,data-2026-05.parquet,574.7,14427217,2026-05-01 00:00:00,2026-05-31 23:59:00,31,5348,85987,51109,545511,3.7811
2026-06,data-2026-06.parquet,589.73,14752336,2026-06-01 00:00:00,2026-06-30 23:59:00,30,5374,87788,51260,897561,6.0842
2026-07,data-2026-07.parquet,563.05,14052153,2026-07-01 00:00:00,2026-07-31 23:59:00,31,5378,85086,51819,669203,4.7623


In [10]:
total_scope_summary_df = (
    historical_2026
    .agg(
        F.count("*").alias("row_count_jan_jul"),
        F.min("time_ts").alias("minimum_time"),
        F.max("time_ts").alias("maximum_time"),
        F.countDistinct(
            F.to_date("time_ts")
        ).alias("distinct_service_dates"),
        F.approx_count_distinct(
            "eva",
            rsd=0.02,
        ).alias("approx_distinct_eva"),
        F.approx_count_distinct(
            "train_line_ride_id",
            rsd=0.02,
        ).alias("approx_distinct_rides"),
    )
)

display_spark_table(
    total_scope_summary_df,
    "Jan-Jul 2026 combined scope",
)


row_count_jan_jul,minimum_time,maximum_time,distinct_service_dates,approx_distinct_eva,approx_distinct_rides
101702091,2026-01-01 00:00:00,2026-07-31 23:59:00,212,5453,195687


## 7. Column completeness — null counts and null rates

A nullable Spark field is not automatically a bad field. This first produces the broad
null profile; the next section investigates whether important null patterns are
structural or suspicious.


In [11]:
null_aggregate_expressions = [
    F.count("*").alias("row_count")
]

for column_name in EXPECTED_COLUMNS:
    null_aggregate_expressions.append(
        F.sum(
            F.when(F.col(column_name).isNull(), 1)
            .otherwise(0)
        ).alias(f"{column_name}__null_count")
    )

null_profile_wide_df = (
    historical_2026_raw
    .groupBy("source_month")
    .agg(*null_aggregate_expressions)
    .orderBy("source_month")
)

null_profile_rows = []

for row in null_profile_wide_df.collect():
    row_dict = row.asDict()
    row_count = row_dict["row_count"]

    for column_name in EXPECTED_COLUMNS:
        null_count = row_dict[f"{column_name}__null_count"]
        null_profile_rows.append(
            (
                row_dict["source_month"],
                column_name,
                row_count,
                null_count,
                round(100.0 * null_count / row_count, 4),
            )
        )

null_profile_df = spark.createDataFrame(
    null_profile_rows,
    (
        "source_month string, "
        "column_name string, "
        "row_count long, "
        "null_count long, "
        "null_rate_pct double"
    ),
)

nonzero_null_profile_df = (
    null_profile_df
    .filter(F.col("null_count") > 0)
    .orderBy(
        "source_month",
        F.desc("null_rate_pct"),
        "column_name",
    )
)

display_spark_table(
    nonzero_null_profile_df,
    "Columns with nulls by month",
    limit=200,
)


source_month,column_name,row_count,null_count,null_rate_pct
2026-01,arrival_planned_time,15582748,1142697,7.3331
2026-01,departure_planned_time,15582748,1142259,7.3303
2026-01,arrival_change_time,15582748,1142241,7.3302
2026-01,departure_change_time,15582748,1141767,7.3271
2026-01,line_number,15582748,279369,1.7928
2026-01,station_name,15582748,2121,0.0136
2026-02,arrival_planned_time,13721520,1029311,7.5014
2026-02,departure_planned_time,13721520,1029052,7.4995
2026-02,arrival_change_time,13721520,1028907,7.4985
2026-02,departure_change_time,13721520,1028585,7.4961


## 8. Context-dependent null patterns

The data catalog already warns that some nulls can be valid:

- arrival fields may be absent at an origin stop;
- departure fields may be absent at a destination stop;
- `line_number` may be absent for some long-distance services;
- `station_name` may be absent when an EVA lookup fails even if the XML name exists.

The patterns below are therefore profiled explicitly before proposing dbt tests.


In [12]:
contextual_null_profile_df = (
    historical_2026_raw
    .groupBy("source_month")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNull()
                & F.col("departure_planned_time").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("arrival_null_departure_present"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNotNull()
                & F.col("departure_planned_time").isNull(),
                1,
            ).otherwise(0)
        ).alias("arrival_present_departure_null"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNull()
                & F.col("departure_planned_time").isNull(),
                1,
            ).otherwise(0)
        ).alias("both_planned_events_null"),
        F.sum(
            F.when(
                F.col("station_name").isNull()
                & F.col("xml_station_name").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("station_lookup_missing_xml_present"),
        F.sum(
            F.when(
                F.col("station_name").isNotNull()
                & F.col("xml_station_name").isNull(),
                1,
            ).otherwise(0)
        ).alias("station_present_xml_missing"),
        F.sum(
            F.when(F.col("line_number").isNull(), 1)
            .otherwise(0)
        ).alias("line_number_null"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNotNull()
                & F.col("arrival_change_time").isNull(),
                1,
            ).otherwise(0)
        ).alias("arrival_planned_present_change_null"),
        F.sum(
            F.when(
                F.col("departure_planned_time").isNotNull()
                & F.col("departure_change_time").isNull(),
                1,
            ).otherwise(0)
        ).alias("departure_planned_present_change_null"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNotNull()
                & (
                    F.col("arrival_planned_time")
                    == F.col("arrival_change_time")
                ),
                1,
            ).otherwise(0)
        ).alias("arrival_change_equals_planned"),
        F.sum(
            F.when(
                F.col("departure_planned_time").isNotNull()
                & (
                    F.col("departure_planned_time")
                    == F.col("departure_change_time")
                ),
                1,
            ).otherwise(0)
        ).alias("departure_change_equals_planned"),
        F.sum(
            F.when(F.col("time").isNull(), 1)
            .otherwise(0)
        ).alias("time_null"),
    )
    .orderBy("source_month")
)

display_spark_table(
    contextual_null_profile_df,
    "Context-dependent null and fallback patterns",
)


source_month,row_count,arrival_null_departure_present,arrival_present_departure_null,both_planned_events_null,station_lookup_missing_xml_present,station_present_xml_missing,line_number_null,arrival_planned_present_change_null,departure_planned_present_change_null,arrival_change_equals_planned,departure_change_equals_planned,time_null
2026-01,15582748,1142697,1142259,0,2121,0,279369,0,0,5622412,5445535,0
2026-02,13721520,1029311,1029052,0,1811,0,242647,0,0,5032519,4868214,0
2026-03,15016329,1132570,1130460,0,2131,0,254543,0,0,5623200,5361309,0
2026-04,14149788,1086580,1078559,0,1888,0,247661,0,0,5316444,5052192,0
2026-05,14427217,1113742,1101700,0,3659,0,261223,0,0,5353773,5076608,0
2026-06,14752336,1139138,1136405,0,4703,0,249048,0,0,5249824,5050676,0
2026-07,14052153,1089808,1100981,0,4708,0,245980,0,0,5085455,4694196,0


## 9. Cardinality and categorical domains

Broad cardinality uses `approx_count_distinct` to avoid unnecessary exact shuffles on
high-cardinality dimensions. The purpose is to understand scale and whether a field is
suitable as a dimension, not to prove uniqueness.


In [13]:
CARDINALITY_COLUMNS = [
    "station_name",
    "xml_station_name",
    "eva",
    "train_number",
    "line_number",
    "final_destination_station",
    "train_type",
    "train_line_ride_id",
]

cardinality_agg = [
    F.count("*").alias("row_count")
]

for column_name in CARDINALITY_COLUMNS:
    cardinality_agg.append(
        F.approx_count_distinct(
            column_name,
            rsd=0.02,
        ).alias(f"{column_name}__approx_distinct")
    )

cardinality_wide_df = (
    historical_2026_raw
    .groupBy("source_month")
    .agg(*cardinality_agg)
    .orderBy("source_month")
)

cardinality_rows = []

for row in cardinality_wide_df.collect():
    row_dict = row.asDict()

    for column_name in CARDINALITY_COLUMNS:
        cardinality_rows.append(
            (
                row_dict["source_month"],
                column_name,
                row_dict[f"{column_name}__approx_distinct"],
            )
        )

cardinality_profile_df = spark.createDataFrame(
    cardinality_rows,
    (
        "source_month string, "
        "column_name string, "
        "approx_distinct_count long"
    ),
).orderBy("source_month", "column_name")

display_spark_table(
    cardinality_profile_df,
    "Approximate distinct values by month",
    limit=200,
)


26/08/09 20:02:37 WARN DAGScheduler: Broadcasting large task binary with size 1886.8 KiB
26/08/09 20:02:37 WARN DAGScheduler: Broadcasting large task binary with size 1887.2 KiB
26/08/09 20:02:37 WARN DAGScheduler: Broadcasting large task binary with size 1996.5 KiB


source_month,column_name,approx_distinct_count
2026-01,eva,5400
2026-01,final_destination_station,2152
2026-01,line_number,320
2026-01,station_name,5352
2026-01,train_line_ride_id,79064
2026-01,train_number,49353
2026-01,train_type,85
2026-01,xml_station_name,5374
2026-02,eva,5400
2026-02,final_destination_station,2217


In [14]:
train_type_counts_df = (
    historical_2026_raw
    .groupBy("source_month", "train_type")
    .agg(F.count("*").alias("row_count"))
)

train_type_window = Window.partitionBy(
    "source_month"
).orderBy(
    F.desc("row_count"),
    F.asc_nulls_last("train_type"),
)

top_train_types_df = (
    train_type_counts_df
    .withColumn(
        "rank_in_month",
        F.row_number().over(train_type_window),
    )
    .filter(F.col("rank_in_month") <= 15)
    .orderBy("source_month", "rank_in_month")
)

display_spark_table(
    top_train_types_df,
    "Top train types by month",
    limit=120,
)


source_month,train_type,row_count,rank_in_month
2026-01,S,7014139,1
2026-01,RB,1906794,2
2026-01,RE,1457699,3
2026-01,HLB,373446,4
2026-01,Bus,281439,5
2026-01,ARV,258658,6
2026-01,BRB,244525,7
2026-01,OE,226361,8
2026-01,ERB,225324,9
2026-01,NWB,223575,10


In [15]:
line_null_by_train_type_df = (
    historical_2026_raw
    .groupBy("train_type")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(F.col("line_number").isNull(), 1)
            .otherwise(0)
        ).alias("line_number_null_count"),
    )
    .withColumn(
        "line_number_null_rate_pct",
        F.round(
            100.0
            * F.col("line_number_null_count")
            / F.col("row_count"),
            4,
        ),
    )
    .orderBy(F.desc("row_count"))
)

display_spark_table(
    line_null_by_train_type_df,
    "Line-number completeness by train type",
    limit=100,
)


train_type,row_count,line_number_null_count,line_number_null_rate_pct
S,45627398,30,0.0001
RB,12557666,341,0.0027
RE,9512485,1123,0.0118
HLB,2378498,0,0.0
Bus,2075396,0,0.0
ARV,1723789,0,0.0
BRB,1581840,0,0.0
NWB,1452152,0,0.0
ERB,1434631,0,0.0
OE,1433305,25,0.0017


## 10. Station identity consistency

`eva` is the source station identifier and should be treated as the primary evidence
for station identity. This section checks whether the same EVA is associated with
multiple resolved or XML station names during the seven-month window.

Multiple names do not automatically mean corruption: spelling changes, source naming,
or mapping updates may explain them. The goal is to discover whether `dim_station`
needs a canonical-name mapping rather than destructively replacing source text.


In [16]:
eva_station_name_profile_df = (
    historical_2026_raw
    .filter(F.col("eva").isNotNull())
    .groupBy("eva")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("station_name").alias(
            "distinct_resolved_station_names"
        ),
        F.countDistinct("xml_station_name").alias(
            "distinct_xml_station_names"
        ),
        F.min("station_name").alias("sample_station_name_min"),
        F.max("station_name").alias("sample_station_name_max"),
        F.min("xml_station_name").alias("sample_xml_name_min"),
        F.max("xml_station_name").alias("sample_xml_name_max"),
    )
)

unstable_eva_names_df = (
    eva_station_name_profile_df
    .filter(
        (F.col("distinct_resolved_station_names") > 1)
        | (F.col("distinct_xml_station_names") > 1)
    )
    .orderBy(
        F.desc("row_count"),
        "eva",
    )
)

print(
    "EVA identifiers with >1 resolved or XML station name:",
    unstable_eva_names_df.count(),
)

display_spark_table(
    unstable_eva_names_df,
    "Sample EVA identifiers with naming variation",
    limit=50,
)


EVA identifiers with >1 resolved or XML station name: 5


eva,row_count,distinct_resolved_station_names,distinct_xml_station_names,sample_station_name_min,sample_station_name_max,sample_xml_name_min,sample_xml_name_max
08003185,13026,1,2,Karlsruhe-Hagsfeld,Karlsruhe-Hagsfeld,Hagsfeld Bahnhof,Karlsruhe-Hagsfeld
08000892,7008,1,2,Berghausen,Berghausen,Berghausen,Berghausen(b Wittgenstein)
08002815,3379,1,2,Oberzent-Hetzbach,Oberzent-Hetzbach,Beerfelden Hetzbach,Oberzent-Hetzbach
08003144,3378,1,2,Oberzent-Kailbach,Oberzent-Kailbach,Hesseneck Kailbach,Oberzent-Kailbach
08005393,3378,1,2,Oberzent-Schöllenbach,Oberzent-Schöllenbach,Hesseneck Schöllenbach,Oberzent-Schöllenbach


In [17]:
station_name_comparison_df = (
    historical_2026_raw
    .groupBy("source_month")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(
                F.col("station_name").isNotNull()
                & F.col("xml_station_name").isNotNull()
                & (
                    F.col("station_name")
                    != F.col("xml_station_name")
                ),
                1,
            ).otherwise(0)
        ).alias("resolved_vs_xml_name_mismatch_rows"),
        F.sum(
            F.when(
                F.col("station_name").isNull()
                & F.col("xml_station_name").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("resolved_missing_xml_present_rows"),
    )
    .withColumn(
        "name_mismatch_rate_pct",
        F.round(
            100.0
            * F.col("resolved_vs_xml_name_mismatch_rows")
            / F.col("row_count"),
            4,
        ),
    )
    .orderBy("source_month")
)

display_spark_table(
    station_name_comparison_df,
    "Resolved station name vs XML station name",
)


source_month,row_count,resolved_vs_xml_name_mismatch_rows,resolved_missing_xml_present_rows,name_mismatch_rate_pct
2026-01,15582748,4366941,2121,28.0242
2026-02,13721520,3851110,1811,28.0662
2026-03,15016329,4247349,2131,28.2849
2026-04,14149788,4002270,1888,28.285
2026-05,14427217,4064443,3659,28.1721
2026-06,14752336,4166426,4703,28.2425
2026-07,14052153,3848271,4708,27.3856


# 11. Grain and identifier investigation

The catalog's current hypothesis is:

> one row represents one processed service-stop observation for one train ride at one
> station sequence position.

Existing catalog evidence showed that `id` was unique in the sampled months, while
`train_line_ride_id + train_line_station_num` repeated. This section extends the exact
key checks across every month from January through July 2026 and investigates what
changes within repeated ride-stop groups.


### 11.1 Exact `id` assessment

This is intentionally exact rather than approximate because `id` is the candidate
source row key.


In [18]:
id_groups_df = (
    historical_2026_raw
    .groupBy("source_month", "id")
    .agg(F.count("*").alias("observations_per_id"))
)

id_key_profile_df = (
    id_groups_df
    .groupBy("source_month")
    .agg(
        F.sum("observations_per_id").alias("row_count"),
        F.count("*").alias("distinct_id_groups_including_null"),
        F.sum(
            F.when(
                F.col("id").isNull(),
                F.col("observations_per_id"),
            ).otherwise(0)
        ).alias("null_id_rows"),
        F.sum(
            F.when(
                F.col("observations_per_id") > 1,
                1,
            ).otherwise(0)
        ).alias("duplicate_id_groups"),
        F.sum(
            F.when(
                F.col("observations_per_id") > 1,
                F.col("observations_per_id") - 1,
            ).otherwise(0)
        ).alias("duplicate_excess_rows"),
        F.max("observations_per_id").alias(
            "max_observations_per_id"
        ),
    )
    .orderBy("source_month")
)

display_spark_table(
    id_key_profile_df,
    "Exact id key assessment",
)


source_month,row_count,distinct_id_groups_including_null,null_id_rows,duplicate_id_groups,duplicate_excess_rows,max_observations_per_id
2026-01,15582748,15582748,0,0,0,1
2026-02,13721520,13721520,0,0,0,1
2026-03,15016329,15016329,0,0,0,1
2026-04,14149788,14149788,0,0,0,1
2026-05,14427217,14427217,0,0,0,1
2026-06,14752336,14752336,0,0,0,1
2026-07,14052153,14052153,0,0,0,1


### 11.2 Repeated ride-stop business keys

`train_line_ride_id + train_line_station_num` identifies a logical stop within a ride,
but it is not assumed to be unique. Repetition is profiled as evidence rather than
removed.

For repeated groups we ask:

- does `time` change?
- does `delay_in_min` change?
- does cancellation status change?
- do arrival/departure effective timestamps change?
- are there multiple technical `id` values?


In [19]:
ride_stop_groups_df = (
    historical_2026_raw
    .groupBy(
        "source_month",
        "train_line_ride_id",
        "train_line_station_num",
    )
    .agg(
        F.count("*").alias("observation_count"),
        F.countDistinct("id").alias("distinct_ids"),
        F.countDistinct("time").alias("distinct_time_values"),
        F.countDistinct("delay_in_min").alias(
            "distinct_delay_values"
        ),
        F.countDistinct("is_canceled").alias(
            "distinct_cancellation_values"
        ),
        F.countDistinct("arrival_change_time").alias(
            "distinct_arrival_change_values"
        ),
        F.countDistinct("departure_change_time").alias(
            "distinct_departure_change_values"
        ),
        F.min("time").alias("minimum_time_raw"),
        F.max("time").alias("maximum_time_raw"),
        F.min("delay_in_min").alias("minimum_delay"),
        F.max("delay_in_min").alias("maximum_delay"),
    )
)

repeated_ride_stop_groups_df = (
    ride_stop_groups_df
    .filter(F.col("observation_count") > 1)
)

ride_stop_duplicate_summary_df = (
    repeated_ride_stop_groups_df
    .groupBy("source_month")
    .agg(
        F.count("*").alias("duplicate_business_key_groups"),
        F.sum("observation_count").alias(
            "rows_in_duplicate_groups"
        ),
        F.sum(
            F.col("observation_count") - 1
        ).alias("extra_rows_beyond_one_per_key"),
        F.max("observation_count").alias(
            "max_observations_per_business_key"
        ),
        F.sum(
            F.when(F.col("distinct_time_values") > 1, 1)
            .otherwise(0)
        ).alias("groups_with_time_changes"),
        F.sum(
            F.when(F.col("distinct_delay_values") > 1, 1)
            .otherwise(0)
        ).alias("groups_with_delay_changes"),
        F.sum(
            F.when(
                F.col("distinct_cancellation_values") > 1,
                1,
            ).otherwise(0)
        ).alias("groups_with_cancellation_changes"),
        F.sum(
            F.when(
                F.col("distinct_arrival_change_values") > 1,
                1,
            ).otherwise(0)
        ).alias("groups_with_arrival_updates"),
        F.sum(
            F.when(
                F.col("distinct_departure_change_values") > 1,
                1,
            ).otherwise(0)
        ).alias("groups_with_departure_updates"),
        F.sum(
            F.when(F.col("distinct_ids") > 1, 1)
            .otherwise(0)
        ).alias("groups_with_multiple_ids"),
    )
    .orderBy("source_month")
)

display_spark_table(
    ride_stop_duplicate_summary_df,
    "Repeated ride-stop business-key summary",
)


source_month,duplicate_business_key_groups,rows_in_duplicate_groups,extra_rows_beyond_one_per_key,max_observations_per_business_key,groups_with_time_changes,groups_with_delay_changes,groups_with_cancellation_changes,groups_with_arrival_updates,groups_with_departure_updates,groups_with_multiple_ids
2026-01,798555,15488778,14690223,32,798555,733642,348388,738086,738334,798555
2026-02,813060,13614083,12801023,29,813060,744593,270460,749623,749609,813060
2026-03,841151,14908587,14067436,35,841151,755682,256178,774476,774399,841151
2026-04,832419,14087164,13254745,31,832419,747503,250830,766937,767409,832419
2026-05,853536,14363141,13509605,32,853536,773040,287693,784658,785367,853536
2026-06,852534,14664281,13811747,31,852534,771933,367715,784794,784645,852534
2026-07,863139,13976135,13112996,32,863139,783908,297475,794544,794914,863139


In [20]:
repeated_ride_stop_samples_df = (
    repeated_ride_stop_groups_df
    .orderBy(
        F.desc("observation_count"),
        F.desc("distinct_delay_values"),
        "source_month",
    )
    .limit(50)
)

display_spark_table(
    repeated_ride_stop_samples_df,
    "High-repetition ride-stop groups to inspect",
    limit=50,
)


source_month,train_line_ride_id,train_line_station_num,observation_count,distinct_ids,distinct_time_values,distinct_delay_values,distinct_cancellation_values,distinct_arrival_change_values,distinct_departure_change_values,minimum_time_raw,maximum_time_raw,minimum_delay,maximum_delay
2026-03,5303821649676246986,1,35,35,33,3,1,0,33,1772406540000000000,1774998540000000000,0,12
2026-05,7696674432816068782,14,32,32,32,30,2,32,30,1777595040000000000,1780269900000000000,-5,212
2026-07,-5885856201096992744,13,32,32,32,29,2,32,0,1782864120000000000,1785535020000000000,-1,147
2026-01,-5040855886902453859,15,32,32,32,28,2,32,32,1767229800000000000,1769901420000000000,-2,115
2026-07,-4383803236760390820,7,32,32,32,28,2,32,28,1782866160000000000,1785542100000000000,-2,109
2026-07,-1854738006003716349,8,32,32,32,27,2,32,31,1782864840000000000,1785541680000000000,0,104
2026-07,-6813803786124510421,6,32,32,32,27,1,32,32,1782866760000000000,1785540360000000000,3,124
2026-07,-1854738006003716349,7,32,32,32,27,2,32,32,1782864240000000000,1785537840000000000,0,81
2026-07,-4737482986446738332,6,32,32,32,27,1,32,30,1782864000000000000,1785540420000000000,0,117
2026-05,7696674432816068782,13,32,32,32,26,2,32,32,1777594200000000000,1780269120000000000,-2,215


### 11.3 Inspect the underlying rows for a chosen repeated ride-stop

Run the cell below after choosing one `train_line_ride_id` and
`train_line_station_num` from the sample table above. It deliberately uses placeholders
instead of automatically choosing a case, because the purpose is human investigation.


In [21]:
# Replace these values with a repeated group from the table above.
INSPECT_RIDE_ID = None
INSPECT_STATION_NUM = None
INSPECT_MONTH = None

if (
    INSPECT_RIDE_ID is not None
    and INSPECT_STATION_NUM is not None
):
    inspected_rows_df = (
        historical_2026
        .filter(
            F.col("train_line_ride_id")
            == F.lit(str(INSPECT_RIDE_ID))
        )
        .filter(
            F.col("train_line_station_num")
            == F.lit(int(INSPECT_STATION_NUM))
        )
    )

    if INSPECT_MONTH is not None:
        inspected_rows_df = inspected_rows_df.filter(
            F.col("source_month") == INSPECT_MONTH
        )

    inspected_rows_df = inspected_rows_df.select(
        "source_month",
        "id",
        "train_line_ride_id",
        "train_line_station_num",
        "station_name",
        "eva",
        "delay_in_min",
        "is_canceled",
        "time_ts",
        "arrival_planned_time_ts",
        "arrival_change_time_ts",
        "departure_planned_time_ts",
        "departure_change_time_ts",
    ).orderBy("time_ts", "id")

    display_spark_table(
        inspected_rows_df,
        "Rows in selected repeated ride-stop group",
        limit=200,
    )
else:
    print(
        "Set INSPECT_RIDE_ID and INSPECT_STATION_NUM "
        "to inspect a repeated group."
    )


Set INSPECT_RIDE_ID and INSPECT_STATION_NUM to inspect a repeated group.


### Optional deep check: semantic duplicates ignoring `id`

This is computationally expensive because it groups on almost the entire row. It is
disabled by default. Enable it only if the ride-stop investigation suggests that
multiple technical IDs may represent identical semantic observations.


In [22]:
RUN_DEEP_EXACT_DUPLICATE_CHECK = False

semantic_duplicate_summary_df = None

if RUN_DEEP_EXACT_DUPLICATE_CHECK:
    semantic_columns = [
        column_name
        for column_name in EXPECTED_COLUMNS
        if column_name != "id"
    ]

    semantic_duplicate_groups_df = (
        historical_2026_raw
        .groupBy("source_month", *semantic_columns)
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
    )

    semantic_duplicate_summary_df = (
        semantic_duplicate_groups_df
        .groupBy("source_month")
        .agg(
            F.count("*").alias("semantic_duplicate_groups"),
            F.sum("row_count").alias(
                "rows_in_semantic_duplicate_groups"
            ),
            F.sum(
                F.col("row_count") - 1
            ).alias("semantic_duplicate_excess_rows"),
        )
        .orderBy("source_month")
    )

    display_spark_table(
        semantic_duplicate_summary_df,
        "Semantic duplicates ignoring id",
    )
else:
    print("Deep semantic duplicate check is disabled.")


Deep semantic duplicate check is disabled.


# 12. Timestamp semantics and integrity

The generic `time` field is documented as the actual arrival or departure time. Before
using it for service date/hour dimensions, this section tests whether it equals the
effective arrival timestamp, effective departure timestamp, both, or neither.


In [23]:
time_match_classified_df = (
    historical_2026_raw
    .withColumn(
        "time_match_category",
        F.when(
            F.col("time").isNull(),
            F.lit("time_null"),
        )
        .when(
            F.col("arrival_change_time").isNotNull()
            & F.col("departure_change_time").isNotNull()
            & (F.col("time") == F.col("arrival_change_time"))
            & (F.col("time") == F.col("departure_change_time")),
            F.lit("matches_both"),
        )
        .when(
            F.col("arrival_change_time").isNotNull()
            & (F.col("time") == F.col("arrival_change_time")),
            F.lit("matches_arrival_only"),
        )
        .when(
            F.col("departure_change_time").isNotNull()
            & (F.col("time") == F.col("departure_change_time")),
            F.lit("matches_departure_only"),
        )
        .otherwise(F.lit("matches_neither")),
    )
)

time_match_counts_df = (
    time_match_classified_df
    .groupBy("source_month", "time_match_category")
    .agg(F.count("*").alias("row_count"))
)

time_month_totals = (
    time_match_counts_df
    .groupBy("source_month")
    .agg(F.sum("row_count").alias("month_row_count"))
)

time_match_profile_df = (
    time_match_counts_df
    .join(time_month_totals, on="source_month")
    .withColumn(
        "rate_pct",
        F.round(
            100.0 * F.col("row_count") / F.col("month_row_count"),
            4,
        ),
    )
    .orderBy("source_month", "time_match_category")
)

display_spark_table(
    time_match_profile_df,
    "How the generic time field relates to event timestamps",
    limit=100,
)


source_month,time_match_category,row_count,month_row_count,rate_pct
2026-01,matches_arrival_only,1141767,15582748,7.3271
2026-01,matches_both,4368836,15582748,28.0364
2026-01,matches_departure_only,10072145,15582748,64.6365
2026-02,matches_arrival_only,1028585,13721520,7.4961
2026-02,matches_both,3801731,13721520,27.7063
2026-02,matches_departure_only,8891204,13721520,64.7975
2026-03,matches_arrival_only,1130166,15016329,7.5262
2026-03,matches_both,4011896,15016329,26.7169
2026-03,matches_departure_only,9874267,15016329,65.7569
2026-04,matches_arrival_only,1078022,14149788,7.6186


### Timestamp relationship checks

These checks are descriptive. Early arrival/departure is valid behavior, while a
negative scheduled dwell (`arrival_planned_time > departure_planned_time`) deserves
inspection. The minute-granularity checks help validate the arithmetic assumptions used
later for delay recomputation.


In [24]:
timestamp_integrity_df = (
    historical_2026_raw
    .groupBy("source_month")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNotNull()
                & F.col("departure_planned_time").isNotNull()
                & (
                    F.col("arrival_planned_time")
                    > F.col("departure_planned_time")
                ),
                1,
            ).otherwise(0)
        ).alias("planned_arrival_after_departure"),
        F.sum(
            F.when(
                F.col("arrival_change_time").isNotNull()
                & F.col("departure_change_time").isNotNull()
                & (
                    F.col("arrival_change_time")
                    > F.col("departure_change_time")
                ),
                1,
            ).otherwise(0)
        ).alias("effective_arrival_after_departure"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNotNull()
                & F.col("arrival_change_time").isNotNull()
                & (
                    F.col("arrival_change_time")
                    < F.col("arrival_planned_time")
                ),
                1,
            ).otherwise(0)
        ).alias("early_arrival_rows"),
        F.sum(
            F.when(
                F.col("departure_planned_time").isNotNull()
                & F.col("departure_change_time").isNotNull()
                & (
                    F.col("departure_change_time")
                    < F.col("departure_planned_time")
                ),
                1,
            ).otherwise(0)
        ).alias("early_departure_rows"),
        F.sum(
            F.when(
                F.col("arrival_planned_time").isNotNull()
                & F.col("arrival_change_time").isNotNull()
                & (
                    F.pmod(
                        F.abs(
                            F.col("arrival_change_time")
                            - F.col("arrival_planned_time")
                        ),
                        F.lit(NS_PER_MINUTE),
                    )
                    != 0
                ),
                1,
            ).otherwise(0)
        ).alias("arrival_deltas_not_whole_minutes"),
        F.sum(
            F.when(
                F.col("departure_planned_time").isNotNull()
                & F.col("departure_change_time").isNotNull()
                & (
                    F.pmod(
                        F.abs(
                            F.col("departure_change_time")
                            - F.col("departure_planned_time")
                        ),
                        F.lit(NS_PER_MINUTE),
                    )
                    != 0
                ),
                1,
            ).otherwise(0)
        ).alias("departure_deltas_not_whole_minutes"),
    )
    .orderBy("source_month")
)

display_spark_table(
    timestamp_integrity_df,
    "Timestamp integrity and granularity checks",
)


source_month,row_count,planned_arrival_after_departure,effective_arrival_after_departure,early_arrival_rows,early_departure_rows,arrival_deltas_not_whole_minutes,departure_deltas_not_whole_minutes
2026-01,15582748,0,77053,285659,28165,0,0
2026-02,13721520,0,66519,300728,31504,0,0
2026-03,15016329,0,70125,355757,29319,0,0
2026-04,14149788,0,49031,365466,22381,0,0
2026-05,14427217,0,51378,348788,22933,0,0
2026-06,14752336,0,77419,341430,21003,0,0
2026-07,14052153,0,60836,332423,17919,0,0


# 13. Delay distribution profiling

The project needs monthly delay reporting, but an average alone can hide a long tail.
This section profiles non-canceled observations with percentiles and descriptive delay
bands.

The bands (`>=5`, `>=15`, `>=30`, etc.) are **profiling bins only**. They are not yet
business definitions of minor/major delay.


In [25]:
non_canceled_delay_df = historical_2026_raw.filter(
    F.col("is_canceled") == False
)

delay_profile_df = (
    non_canceled_delay_df
    .groupBy("source_month")
    .agg(
        F.count("*").alias("non_canceled_rows"),
        F.count("delay_in_min").alias("delay_non_null_rows"),
        F.sum(
            F.when(F.col("delay_in_min").isNull(), 1)
            .otherwise(0)
        ).alias("delay_null_rows"),
        F.min("delay_in_min").alias("minimum_delay_min"),
        F.max("delay_in_min").alias("maximum_delay_min"),
        F.round(F.avg("delay_in_min"), 4).alias(
            "average_delay_min"
        ),
        F.round(F.stddev("delay_in_min"), 4).alias(
            "stddev_delay_min"
        ),
        F.percentile_approx(
            F.col("delay_in_min"),
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
            10000,
        ).alias("delay_percentiles_p01_p05_p25_p50_p75_p90_p95_p99"),
        F.sum(
            F.when(F.col("delay_in_min") < 0, 1)
            .otherwise(0)
        ).alias("negative_delay_rows"),
        F.sum(
            F.when(F.col("delay_in_min") == 0, 1)
            .otherwise(0)
        ).alias("zero_delay_rows"),
        *[
            F.sum(
                F.when(
                    F.col("delay_in_min") >= threshold,
                    1,
                ).otherwise(0)
            ).alias(f"delay_ge_{threshold}_min_rows")
            for threshold in [5, 15, 30, 60, 120, 300]
        ],
    )
    .orderBy("source_month")
)

display_spark_table(
    delay_profile_df,
    "Non-canceled delay distribution by month",
)


source_month,non_canceled_rows,delay_non_null_rows,delay_null_rows,minimum_delay_min,maximum_delay_min,average_delay_min,stddev_delay_min,delay_percentiles_p01_p05_p25_p50_p75_p90_p95_p99,negative_delay_rows,zero_delay_rows,delay_ge_5_min_rows,delay_ge_15_min_rows,delay_ge_30_min_rows,delay_ge_60_min_rows,delay_ge_120_min_rows,delay_ge_300_min_rows
2026-01,14678562,14678562,0,-1440,802,3.3524,7.6605,"[0, 0, 0, 1, 3, 8, 14, 34]",72656,5230777,2923402,706352,200838,38102,4960,167
2026-02,13217739,13217739,0,-120,900,2.8944,6.4983,"[0, 0, 0, 1, 3, 7, 12, 30]",77464,4948614,2308278,500432,131749,22182,2538,40
2026-03,14552068,14552068,0,-130,1439,2.8844,7.937,"[0, 0, 0, 1, 3, 7, 12, 30]",82719,5531440,2497819,569790,147129,21865,2430,339
2026-04,13702759,13702759,0,-1446,1356,2.8221,6.3089,"[0, 0, 0, 1, 3, 7, 12, 29]",82403,5208966,2317317,521028,132206,20056,1994,13
2026-05,13881706,13881706,0,-129,1413,3.0658,7.229,"[0, 0, 0, 1, 3, 8, 13, 33]",78634,5151477,2476013,598578,170931,31359,3915,83
2026-06,13854775,13854775,0,-120,1421,3.6248,8.872,"[0, 0, 0, 1, 4, 9, 15, 39]",73573,4871900,2839896,753503,237809,52595,8548,280
2026-07,13382950,13382950,0,-1441,1380,3.3016,7.5707,"[0, 0, 0, 1, 3, 8, 14, 34]",72821,4653793,2604175,627368,182985,35490,3824,183


## 14. Validate delay recomputation against published `delay_in_min`

This is a critical architecture question. The current project plan proposes computing
delay from planned vs. effective event timestamps so both ingestion paths share one
semantic definition. That rule should only be adopted if it reproduces the published
historical value.

Instead of assuming whether `time` represents arrival or departure, the notebook tests
both event-specific candidates:

- arrival delay = `arrival_change_time - arrival_planned_time`;
- departure delay = `departure_change_time - departure_planned_time`.

A row is eligible for an event-specific comparison only when `time` equals that event's
effective timestamp and both planned/effective values are present.


In [26]:
delay_recompute_rows_df = (
    historical_2026_raw
    .withColumn(
        "arrival_delay_calc_min",
        F.when(
            F.col("arrival_planned_time").isNotNull()
            & F.col("arrival_change_time").isNotNull(),
            (
                F.col("arrival_change_time")
                - F.col("arrival_planned_time")
            ) / F.lit(NS_PER_MINUTE),
        ),
    )
    .withColumn(
        "departure_delay_calc_min",
        F.when(
            F.col("departure_planned_time").isNotNull()
            & F.col("departure_change_time").isNotNull(),
            (
                F.col("departure_change_time")
                - F.col("departure_planned_time")
            ) / F.lit(NS_PER_MINUTE),
        ),
    )
    .withColumn(
        "arrival_candidate_eligible",
        F.col("time").isNotNull()
        & F.col("arrival_planned_time").isNotNull()
        & F.col("arrival_change_time").isNotNull()
        & (F.col("time") == F.col("arrival_change_time")),
    )
    .withColumn(
        "departure_candidate_eligible",
        F.col("time").isNotNull()
        & F.col("departure_planned_time").isNotNull()
        & F.col("departure_change_time").isNotNull()
        & (F.col("time") == F.col("departure_change_time")),
    )
    .withColumn(
        "arrival_matches_published",
        F.col("arrival_candidate_eligible")
        & F.col("delay_in_min").isNotNull()
        & (
            F.abs(
                F.col("arrival_delay_calc_min")
                - F.col("delay_in_min")
            ) < F.lit(1e-9)
        ),
    )
    .withColumn(
        "departure_matches_published",
        F.col("departure_candidate_eligible")
        & F.col("delay_in_min").isNotNull()
        & (
            F.abs(
                F.col("departure_delay_calc_min")
                - F.col("delay_in_min")
            ) < F.lit(1e-9)
        ),
    )
    .withColumn(
        "any_event_candidate_eligible",
        F.col("arrival_candidate_eligible")
        | F.col("departure_candidate_eligible"),
    )
    .withColumn(
        "any_event_matches_published",
        F.col("arrival_matches_published")
        | F.col("departure_matches_published"),
    )
)


In [27]:
delay_recompute_validation_df = (
    delay_recompute_rows_df
    .groupBy("source_month")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.col("arrival_candidate_eligible").cast("long")
        ).alias("arrival_eligible_rows"),
        F.sum(
            F.col("arrival_matches_published").cast("long")
        ).alias("arrival_match_rows"),
        F.sum(
            F.col("departure_candidate_eligible").cast("long")
        ).alias("departure_eligible_rows"),
        F.sum(
            F.col("departure_matches_published").cast("long")
        ).alias("departure_match_rows"),
        F.sum(
            F.col("any_event_candidate_eligible").cast("long")
        ).alias("any_event_eligible_rows"),
        F.sum(
            F.col("any_event_matches_published").cast("long")
        ).alias("any_event_match_rows"),
        F.sum(
            F.when(
                F.col("arrival_candidate_eligible")
                & F.col("departure_candidate_eligible"),
                1,
            ).otherwise(0)
        ).alias("rows_where_time_matches_both_events"),
        F.sum(
            F.when(
                F.col("arrival_candidate_eligible")
                & F.col("departure_candidate_eligible")
                & (
                    F.abs(
                        F.col("arrival_delay_calc_min")
                        - F.col("departure_delay_calc_min")
                    ) < F.lit(1e-9)
                ),
                1,
            ).otherwise(0)
        ).alias("both_event_rows_with_same_calculated_delay"),
    )
    .withColumn(
        "arrival_match_rate_pct",
        F.when(
            F.col("arrival_eligible_rows") > 0,
            F.round(
                100.0
                * F.col("arrival_match_rows")
                / F.col("arrival_eligible_rows"),
                4,
            ),
        ),
    )
    .withColumn(
        "departure_match_rate_pct",
        F.when(
            F.col("departure_eligible_rows") > 0,
            F.round(
                100.0
                * F.col("departure_match_rows")
                / F.col("departure_eligible_rows"),
                4,
            ),
        ),
    )
    .withColumn(
        "any_event_match_rate_pct",
        F.when(
            F.col("any_event_eligible_rows") > 0,
            F.round(
                100.0
                * F.col("any_event_match_rows")
                / F.col("any_event_eligible_rows"),
                4,
            ),
        ),
    )
    .orderBy("source_month")
)

display_spark_table(
    delay_recompute_validation_df,
    "Published delay vs event-time recomputation",
)


source_month,row_count,arrival_eligible_rows,arrival_match_rows,departure_eligible_rows,departure_match_rows,any_event_eligible_rows,any_event_match_rows,rows_where_time_matches_both_events,both_event_rows_with_same_calculated_delay,arrival_match_rate_pct,departure_match_rate_pct,any_event_match_rate_pct
2026-01,15582748,5510527,3838052,14440489,14440489,15582321,15582321,4368695,2696220,69.6495,100.0,100.0
2026-02,13721520,4830215,3372778,12692468,12692468,13721125,13721125,3801558,2344121,69.8267,100.0,100.0
2026-03,15016329,5142000,3608981,13885869,13885869,15016084,15016084,4011785,2478766,70.1863,100.0,100.0
2026-04,14149788,4748513,3363473,13071229,13071229,14149369,14149369,3670373,2285333,70.8321,100.0,100.0
2026-05,14427217,4831908,3446641,13325517,13325517,14426975,14426975,3730450,2345183,71.3308,100.0,100.0
2026-06,14752336,5026893,3574795,13615931,13615931,14752078,14752078,3890746,2438648,71.1134,100.0,100.0
2026-07,14052153,4549641,3277765,12951172,12951172,14051977,14051977,3448836,2176960,72.0445,100.0,100.0


In [28]:
delay_recompute_mismatches_df = (
    delay_recompute_rows_df
    .filter(F.col("any_event_candidate_eligible"))
    .filter(~F.col("any_event_matches_published"))
    .select(
        "source_month",
        "id",
        "station_name",
        "eva",
        "train_type",
        "train_number",
        "line_number",
        "delay_in_min",
        "is_canceled",
        "time",
        "arrival_planned_time",
        "arrival_change_time",
        "arrival_delay_calc_min",
        "departure_planned_time",
        "departure_change_time",
        "departure_delay_calc_min",
        "arrival_candidate_eligible",
        "departure_candidate_eligible",
    )
    .orderBy("source_month", "id")
)

print(
    "Rows eligible for event-based delay comparison but not matching "
    "published delay:",
    delay_recompute_mismatches_df.count(),
)

display_spark_table(
    delay_recompute_mismatches_df,
    "Sample delay recomputation mismatches",
    limit=100,
)


Rows eligible for event-based delay comparison but not matching published delay: 0


source_month,id,station_name,eva,train_type,train_number,line_number,delay_in_min,is_canceled,time,arrival_planned_time,arrival_change_time,arrival_delay_calc_min,departure_planned_time,departure_change_time,departure_delay_calc_min,arrival_candidate_eligible,departure_candidate_eligible


## 15. Cancellation profiling

Cancellation must be treated separately from ordinary delay because a canceled
observation should not automatically contribute to the same delay denominator as a
completed stop. This section quantifies cancellation rates and the associated null/time
patterns; it does not decide the final mart rule.


In [29]:
cancellation_profile_counts_df = (
    historical_2026_raw
    .groupBy("source_month", "is_canceled")
    .agg(
        F.count("*").alias("row_count"),
        F.count("delay_in_min").alias("delay_non_null_rows"),
        F.sum(
            F.when(F.col("delay_in_min").isNull(), 1)
            .otherwise(0)
        ).alias("delay_null_rows"),
        F.round(F.avg("delay_in_min"), 4).alias(
            "average_delay_min"
        ),
        F.sum(
            F.when(F.col("time").isNull(), 1)
            .otherwise(0)
        ).alias("time_null_rows"),
        F.sum(
            F.when(F.col("arrival_planned_time").isNull(), 1)
            .otherwise(0)
        ).alias("arrival_planned_null_rows"),
        F.sum(
            F.when(F.col("departure_planned_time").isNull(), 1)
            .otherwise(0)
        ).alias("departure_planned_null_rows"),
    )
)

cancellation_month_totals = (
    cancellation_profile_counts_df
    .groupBy("source_month")
    .agg(F.sum("row_count").alias("month_row_count"))
)

cancellation_profile_df = (
    cancellation_profile_counts_df
    .join(cancellation_month_totals, on="source_month")
    .withColumn(
        "status_rate_pct",
        F.round(
            100.0 * F.col("row_count") / F.col("month_row_count"),
            4,
        ),
    )
    .orderBy("source_month", "is_canceled")
)

display_spark_table(
    cancellation_profile_df,
    "Cancellation status and associated completeness",
    limit=50,
)


source_month,is_canceled,row_count,delay_non_null_rows,delay_null_rows,average_delay_min,time_null_rows,arrival_planned_null_rows,departure_planned_null_rows,month_row_count,status_rate_pct
2026-01,False,14678562,14678562,0,3.3524,0,1074407,1068828,15582748,94.1975
2026-01,True,904186,904186,0,5.0129,0,68290,73431,15582748,5.8025
2026-02,False,13217739,13217739,0,2.8944,0,991181,987217,13721520,96.3285
2026-02,True,503781,503781,0,6.01,0,38130,41835,13721520,3.6715
2026-03,False,14552068,14552068,0,2.8844,0,1096547,1090983,15016329,96.9083
2026-03,True,464261,464261,0,6.9027,0,36023,39477,15016329,3.0917
2026-04,False,13702759,13702759,0,2.8221,0,1054468,1042336,14149788,96.8407
2026-04,True,447029,447029,0,6.4041,0,32112,36223,14149788,3.1593
2026-05,False,13881706,13881706,0,3.0658,0,1074488,1057332,14427217,96.2189
2026-05,True,545511,545511,0,6.4901,0,39254,44368,14427217,3.7811


# 16. Temporal reporting-dimension validation

The future dbt enrichment model is expected to derive service date, hour, weekday, and
month. Here those fields are created only for profiling to verify that they behave as
expected before they become semantic warehouse columns.


In [30]:
temporal_profile_df = (
    historical_2026
    .withColumn(
        "service_date_profile",
        F.to_date("time_ts"),
    )
    .withColumn(
        "service_hour_profile",
        F.hour("time_ts"),
    )
    .withColumn(
        "service_weekday_num_profile",
        F.dayofweek("time_ts"),
    )
    .withColumn(
        "service_weekday_name_profile",
        F.date_format("time_ts", "E"),
    )
)

daily_coverage_df = (
    temporal_profile_df
    .groupBy(
        "source_month",
        "service_date_profile",
    )
    .agg(
        F.count("*").alias("row_count"),
        F.approx_count_distinct(
            "eva",
            rsd=0.02,
        ).alias("approx_distinct_eva"),
        F.sum(
            F.when(F.col("is_canceled") == True, 1)
            .otherwise(0)
        ).alias("canceled_rows"),
    )
)

daily_coverage_summary_df = (
    daily_coverage_df
    .groupBy("source_month")
    .agg(
        F.countDistinct("service_date_profile").alias(
            "distinct_service_dates"
        ),
        F.min("row_count").alias("minimum_daily_rows"),
        F.round(F.avg("row_count"), 2).alias("average_daily_rows"),
        F.max("row_count").alias("maximum_daily_rows"),
        F.min("approx_distinct_eva").alias(
            "minimum_daily_approx_eva"
        ),
        F.max("approx_distinct_eva").alias(
            "maximum_daily_approx_eva"
        ),
    )
    .orderBy("source_month")
)

display_spark_table(
    daily_coverage_summary_df,
    "Daily coverage stability within each month",
)


source_month,distinct_service_dates,minimum_daily_rows,average_daily_rows,maximum_daily_rows,minimum_daily_approx_eva,maximum_daily_approx_eva
2026-01,31,417039,502669.29,537417,5348,5396
2026-02,28,390999,490054.29,529409,5293,5381
2026-03,31,372170,484397.71,519291,5217,5348
2026-04,30,297120,471659.6,514220,5250,5337
2026-05,31,394823,465394.1,511347,5239,5304
2026-06,30,402539,491744.53,531869,5224,5330
2026-07,31,360409,453295.26,517669,5087,5271


In [31]:
service_date_scope_check_df = (
    temporal_profile_df
    .groupBy("source_month")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(F.col("service_date_profile").isNull(), 1)
            .otherwise(0)
        ).alias("service_date_null_rows"),
        F.sum(
            F.when(
                F.col("service_date_profile").isNotNull()
                & (
                    F.date_format(
                        F.col("service_date_profile"),
                        "yyyy-MM",
                    )
                    != F.col("source_month")
                ),
                1,
            ).otherwise(0)
        ).alias("service_date_outside_source_month_rows"),
    )
    .orderBy("source_month")
)

display_spark_table(
    service_date_scope_check_df,
    "Service-date scope check",
)


source_month,row_count,service_date_null_rows,service_date_outside_source_month_rows
2026-01,15582748,0,0
2026-02,13721520,0,0
2026-03,15016329,0,0
2026-04,14149788,0,0
2026-05,14427217,0,0
2026-06,14752336,0,0
2026-07,14052153,0,0


In [32]:
hour_coverage_df = (
    temporal_profile_df
    .groupBy(
        "source_month",
        "service_hour_profile",
    )
    .agg(F.count("*").alias("row_count"))
    .orderBy(
        "source_month",
        "service_hour_profile",
    )
)

weekday_coverage_df = (
    temporal_profile_df
    .groupBy(
        "source_month",
        "service_weekday_num_profile",
        "service_weekday_name_profile",
    )
    .agg(F.count("*").alias("row_count"))
    .orderBy(
        "source_month",
        "service_weekday_num_profile",
    )
)

display_spark_table(
    hour_coverage_df,
    "Hourly observation coverage",
    limit=200,
)

display_spark_table(
    weekday_coverage_df,
    "Weekday observation coverage",
    limit=100,
)


source_month,service_hour_profile,row_count
2026-01,0,349932
2026-01,1,161244
2026-01,2,69685
2026-01,3,62221
2026-01,4,227200
2026-01,5,570155
2026-01,6,769607
2026-01,7,849838
2026-01,8,836262
2026-01,9,808117


source_month,service_weekday_num_profile,service_weekday_name_profile,row_count
2026-01,1,Sun,1706716
2026-01,2,Mon,2132743
2026-01,3,Tue,2097627
2026-01,4,Wed,2126808
2026-01,5,Thu,2567791
2026-01,6,Fri,2668477
2026-01,7,Sat,2282586
2026-02,1,Sun,1652942
2026-02,2,Mon,1952426
2026-02,3,Tue,2083154


# 17. Profiling evidence → candidate dbt requirements

The table below is a decision register, not an automatically enforced rule set. After
reviewing the outputs above, update the `decision_after_profiling` field with the
observed conclusion before implementing dbt models/tests.

This keeps the architecture boundary explicit:

**Spark notebook discovers the rule → dbt owns the production semantic rule/test.**


In [33]:
dbt_requirement_rows = [
    (
        "source row key",
        "Is id non-null and unique in every month?",
        "id_key_profile_df",
        "To decide after profiling",
        "stg_monthly_observations",
        "Candidate: not_null + unique on id",
    ),
    (
        "business grain",
        "Why does train_line_ride_id + train_line_station_num repeat?",
        "ride_stop_duplicate_summary_df + inspected rows",
        "To decide after profiling",
        "int_observations_deduped",
        "Candidate: unique business grain after documented dedup rule",
    ),
    (
        "dedup precedence",
        "Do repeated ride-stop observations behave like updates/snapshots?",
        "repeated_ride_stop_groups_df",
        "To decide after profiling",
        "int_observations_deduped",
        "Candidate: deterministic precedence + duplicate-resolution assertion",
    ),
    (
        "service timestamp",
        "Does time map to arrival, departure, both, or neither?",
        "time_match_profile_df",
        "To decide after profiling",
        "int_observations_enriched",
        "Candidate: service timestamp/date/hour derivation tests",
    ),
    (
        "delay calculation",
        "Can published delay_in_min be reproduced from event timestamps?",
        "delay_recompute_validation_df",
        "To decide after profiling",
        "int_observations_enriched",
        "Candidate: recomputation equality / divergence assertion",
    ),
    (
        "delay sanity",
        "What negative and extreme delay values occur?",
        "delay_profile_df",
        "To decide after profiling",
        "int_observations_enriched",
        "Candidate: evidence-based sanity bounds, not guessed thresholds",
    ),
    (
        "cancellation semantics",
        "Should canceled observations contribute to delay denominators?",
        "cancellation_profile_df",
        "To decide after profiling",
        "mart_daily_* / mart_monthly_*",
        "Candidate: mart-level denominator assertion",
    ),
    (
        "station identity",
        "Does one EVA map to multiple station names?",
        "unstable_eva_names_df",
        "To decide after profiling",
        "dim_station",
        "Candidate: unique EVA in dimension + source-name mapping history",
    ),
    (
        "line completeness",
        "Is a null line_number valid for specific train types?",
        "line_null_by_train_type_df",
        "To decide after profiling",
        "dim_train_line / staging",
        "Avoid blanket not_null until evidence supports it",
    ),
    (
        "temporal dimensions",
        "Do service dates stay inside the source month and cover all days?",
        "service_date_scope_check_df + daily_coverage_summary_df",
        "To decide after profiling",
        "int_observations_enriched",
        "Candidate: accepted date range / non-null derived date tests",
    ),
    (
        "coverage era",
        "Jan-Jul 2026 is post-2025-11-02; how will old history be marked?",
        "Project metadata, not inferred from this seven-month sample",
        "Keep documented transition date",
        "int_observations_enriched",
        "Accepted values for station_coverage_era",
    ),
]

dbt_requirements_df = spark.createDataFrame(
    dbt_requirement_rows,
    (
        "area string, "
        "profiling_question string, "
        "profiling_evidence string, "
        "decision_after_profiling string, "
        "candidate_dbt_model string, "
        "candidate_dbt_test_or_rule string"
    ),
)

display_spark_table(
    dbt_requirements_df,
    "Profiling-to-dbt decision register",
    limit=100,
)


area,profiling_question,profiling_evidence,decision_after_profiling,candidate_dbt_model,candidate_dbt_test_or_rule
source row key,Is id non-null and unique in every month?,id_key_profile_df,To decide after profiling,stg_monthly_observations,Candidate: not_null + unique on id
business grain,Why does train_line_ride_id + train_line_station_num repeat?,ride_stop_duplicate_summary_df + inspected rows,To decide after profiling,int_observations_deduped,Candidate: unique business grain after documented dedup rule
dedup precedence,Do repeated ride-stop observations behave like updates/snapshots?,repeated_ride_stop_groups_df,To decide after profiling,int_observations_deduped,Candidate: deterministic precedence + duplicate-resolution assertion
service timestamp,"Does time map to arrival, departure, both, or neither?",time_match_profile_df,To decide after profiling,int_observations_enriched,Candidate: service timestamp/date/hour derivation tests
delay calculation,Can published delay_in_min be reproduced from event timestamps?,delay_recompute_validation_df,To decide after profiling,int_observations_enriched,Candidate: recomputation equality / divergence assertion
delay sanity,What negative and extreme delay values occur?,delay_profile_df,To decide after profiling,int_observations_enriched,"Candidate: evidence-based sanity bounds, not guessed thresholds"
cancellation semantics,Should canceled observations contribute to delay denominators?,cancellation_profile_df,To decide after profiling,mart_daily_* / mart_monthly_*,Candidate: mart-level denominator assertion
station identity,Does one EVA map to multiple station names?,unstable_eva_names_df,To decide after profiling,dim_station,Candidate: unique EVA in dimension + source-name mapping history
line completeness,Is a null line_number valid for specific train types?,line_null_by_train_type_df,To decide after profiling,dim_train_line / staging,Avoid blanket not_null until evidence supports it
temporal dimensions,Do service dates stay inside the source month and cover all days?,service_date_scope_check_df + daily_coverage_summary_df,To decide after profiling,int_observations_enriched,Candidate: accepted date range / non-null derived date tests


# 19. Export of profiling artifacts

The exports below contain **aggregated profiling evidence**, not transformed source
records. They provide a reproducible record of what was observed when the future dbt
rules were designed.

The full repeated ride-stop group table is not exported by default because it can be
large; export it separately only if needed for deeper analysis.


In [34]:
PROFILE_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

profiling_outputs = {
    "schema_month_summary": schema_month_summary_df,
    "schema_field_comparison": schema_field_comparison_df,
    "monthly_overview": monthly_overview_df,
    "null_profile": null_profile_df,
    "contextual_null_profile": contextual_null_profile_df,
    "cardinality_profile": cardinality_profile_df,
    "line_null_by_train_type": line_null_by_train_type_df,
    "station_name_comparison": station_name_comparison_df,
    "id_key_profile": id_key_profile_df,
    "ride_stop_duplicate_summary": ride_stop_duplicate_summary_df,
    "time_match_profile": time_match_profile_df,
    "timestamp_integrity": timestamp_integrity_df,
    "delay_profile": delay_profile_df,
    "delay_recompute_validation": delay_recompute_validation_df,
    "cancellation_profile": cancellation_profile_df,
    "daily_coverage_summary": daily_coverage_summary_df,
    "service_date_scope_check": service_date_scope_check_df,
    "dbt_requirements": dbt_requirements_df,
}

for output_name, output_df in profiling_outputs.items():
    parquet_path = str(
        PROFILE_OUTPUT_PATH / f"{output_name}_parquet"
    )
    csv_path = str(
        PROFILE_OUTPUT_PATH / f"{output_name}_csv"
    )

    (
        output_df.write
        .mode("overwrite")
        .parquet(parquet_path)
    )

    # The CSV writer rejects array-typed columns (e.g. delay percentiles);
    # flatten them to a comma-joined string for CSV only — Parquet keeps the real array type.
    csv_ready_df = output_df.select(
        *[
            F.concat_ws(",", F.col(field.name)).alias(field.name)
            if field.dataType.typeName() == "array"
            else F.col(field.name)
            for field in output_df.schema.fields
        ]
    )

    (
        csv_ready_df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(csv_path)
    )

    print(f"Saved {output_name}")
    print("  Parquet:", parquet_path)
    print("  CSV:", csv_path)


Saved schema_month_summary
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/schema_month_summary_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/schema_month_summary_csv
Saved schema_field_comparison
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/schema_field_comparison_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/schema_field_comparison_csv


26/08/09 20:17:46 WARN DAGScheduler: Broadcasting large task binary with size 1257.9 KiB
26/08/09 20:17:47 WARN DAGScheduler: Broadcasting large task binary with size 1258.3 KiB
26/08/09 20:17:47 WARN DAGScheduler: Broadcasting large task binary with size 1513.9 KiB
26/08/09 20:17:53 WARN DAGScheduler: Broadcasting large task binary with size 1257.9 KiB
26/08/09 20:17:54 WARN DAGScheduler: Broadcasting large task binary with size 1258.3 KiB
26/08/09 20:17:54 WARN DAGScheduler: Broadcasting large task binary with size 1514.5 KiB


Saved monthly_overview
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/monthly_overview_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/monthly_overview_csv
Saved null_profile
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/null_profile_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/null_profile_csv


Saved contextual_null_profile
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/contextual_null_profile_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/contextual_null_profile_csv
Saved cardinality_profile
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/cardinality_profile_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/cardinality_profile_csv


Saved line_null_by_train_type
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/line_null_by_train_type_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/line_null_by_train_type_csv


...

Saved delay_recompute_validation
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/delay_recompute_validation_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/delay_recompute_validation_csv


Saved cancellation_profile
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/cancellation_profile_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/cancellation_profile_csv


Saved daily_coverage_summary
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/daily_coverage_summary_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/daily_coverage_summary_csv


Saved service_date_scope_check
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/service_date_scope_check_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/service_date_scope_check_csv
Saved dbt_requirements
  Parquet: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/dbt_requirements_parquet
  CSV: /opt/spark/work-dir/artifacts/profiling/deutsche_bahn_2026_01_07/dbt_requirements_csv


26/08/09 21:06:25 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.2: worker lost: Not receiving heartbeat for 60 seconds
26/08/09 21:13:03 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.2: worker lost: Not receiving heartbeat for 60 seconds
